# AIC365 Natural Language Processing — Lab 5
## FA23-BAI-006 | N-gram Language Models & Smoothing

**Scope:** Stage v (Verify) — Home Activities only.

The manual refers to `product_reviews_corpus` and a held-out sentence from earlier activities, but the dataset file itself was not attached. This notebook uses a small sample product-review corpus and the held-out sentence from the lab's worked example. Replace the sample corpus with the official dataset for course submission.

### Setup — sample corpus and tokenization

In [ ]:
import re, math
from collections import Counter, defaultdict
import pandas as pd

product_reviews_corpus = [
    "the camera quality exceeded my expectations",
    "the camera takes clear photos and the battery lasts long",
    "this phone has excellent screen quality",
    "the battery life is poor and the camera is disappointing",
    "excellent product with wonderful build quality",
    "the screen is bright and the camera is sharp",
    "poor battery and disappointing performance",
    "the product exceeded expectations and works well",
    "wonderful camera but the screen is small",
    "the phone is reliable and the battery is good",
    "camera quality is excellent for the price",
    "the product is disappointing and the screen is dull"
]
held_out_sentence = "the movie was fantastic and the acting was convincing"

def tokenize(text):
    return re.findall(r"[a-z]+|[.!?]", text.lower())

tokenized_corpus = [tokenize(s) for s in product_reviews_corpus]
print("Number of sample reviews:", len(tokenized_corpus))
print("Held-out sentence:", held_out_sentence)

### Helper functions — n-gram counts, smoothing and perplexity

In [ ]:
from nltk.lm import MLE, Laplace, WittenBellInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.util import pad_both_ends, ngrams

def train_lm(order, smoothing):
    train_data, vocab = padded_everygram_pipeline(order, tokenized_corpus)
    if smoothing == "mle":
        model = MLE(order)
    elif smoothing == "laplace":
        model = Laplace(order)
    elif smoothing == "witten_bell":
        model = WittenBellInterpolated(order)
    else:
        raise ValueError("Unknown smoothing")
    model.fit(train_data, vocab)
    return model

def sentence_perplexity(model, sentence, order):
    tokens = tokenize(sentence)
    padded = list(pad_both_ends(tokens, n=order))
    test_ngrams = list(ngrams(padded, order))
    # NLTK model.perplexity expects n-grams, not raw tokens.
    return float(model.perplexity(test_ngrams))

models = {
    "Laplace": train_lm(2, "laplace"),
    "Witten-Bell": train_lm(2, "witten_bell")
}

### Activity 1 — Compare two smoothing methods
Fit two smoothed bigram models and compare held-out perplexity.

In [ ]:
perplexities = []
for name, model in models.items():
    try:
        ppl = sentence_perplexity(model, held_out_sentence, 2)
    except Exception as exc:
        ppl = float("inf")
        print(f"{name}: perplexity could not be computed: {exc}")
    perplexities.append({"smoothing": name, "perplexity": ppl})

ppl_df = pd.DataFrame(perplexities)
display(ppl_df)
finite = ppl_df[ppl_df["perplexity"].apply(math.isfinite)]
if not finite.empty:
    best_smoothing = finite.sort_values("perplexity").iloc[0]["smoothing"]
    best_model = models[best_smoothing]
    print("Lower perplexity on this sample:", best_smoothing)
else:
    best_smoothing, best_model = "Witten-Bell", models["Witten-Bell"]
    print("Perplexity comparison was not finite for this small sample; using Witten-Bell for the next activity.")

### Activity 2 — Next-word predictor
Rank the candidates after “the camera quality exceeded my”.

In [ ]:
prefix = "the camera quality exceeded my"
candidates = ["expectations", "budget", "screen"]
context = tokenize(prefix)[-(2-1):]
candidate_rows = []
for word in candidates:
    probability = best_model.score(word, context)
    candidate_rows.append({"candidate": word, "probability": probability})
candidate_df = pd.DataFrame(candidate_rows).sort_values("probability", ascending=False).reset_index(drop=True)
display(candidate_df)
print("Suggested first:", candidate_df.iloc[0]["candidate"])

### Activity 3 — Bigram vs trigram perplexity
Train models with the chosen smoothing method and compare their perplexity.

In [ ]:
def train_selected(order, selected):
    train_data, vocab = padded_everygram_pipeline(order, tokenized_corpus)
    model = Laplace(order) if selected == "Laplace" else WittenBellInterpolated(order)
    model.fit(train_data, vocab)
    return model

bigram_model = train_selected(2, best_smoothing)
trigram_model = train_selected(3, best_smoothing)

order_results = []
for order, model in [(2, bigram_model), (3, trigram_model)]:
    try:
        value = sentence_perplexity(model, held_out_sentence, order)
    except Exception as exc:
        value = float("inf")
        print(f"Could not compute order {order} perplexity: {exc}")
    order_results.append({"model": f"{order}-gram", "perplexity": value})

display(pd.DataFrame(order_results))
print("Interpretation: compare the actual perplexities above; lower perplexity means the model assigns higher probability to this held-out sequence.")

### Activity 4 — OOV word: unsmoothed vs smoothed
Identify a held-out word absent from the training corpus and compare sentence perplexity for MLE and the best smoothed model.

In [ ]:
training_vocab = set(w for sent in tokenized_corpus for w in sent)
held_tokens = tokenize(held_out_sentence)
oov_words = [w for w in held_tokens if w not in training_vocab]
print("Out-of-vocabulary words in held-out sentence:", oov_words)

mle_model = train_lm(2, "mle")
comparison = []
for name, model in [("Unsmoothed MLE", mle_model), (f"Smoothed ({best_smoothing})", best_model)]:
    try:
        ppl = sentence_perplexity(model, held_out_sentence, 2)
    except Exception:
        ppl = float("inf")
    comparison.append({"model": name, "sentence_perplexity": ppl})
display(pd.DataFrame(comparison))
print("An unsmoothed model can assign zero probability to unseen events; this makes sentence perplexity infinite.")

### Note
Because the official `product_reviews_corpus` file is not included in the supplied PDF, numeric results from this notebook are based on the sample reviews above. For official results, replace `product_reviews_corpus` with the instructor-provided corpus and keep the same held-out sentence and evaluation method.